In [ ]:
# 1. Mount Google Drive, install export dependencies, and enter HF_TOKEN
!pip uninstall -y -q torchao
!pip -q install transformers peft accelerate sentencepiece protobuf gguf
from google.colab import drive
drive.mount('/content/drive')
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('Enter HF_TOKEN (for google/gemma-2-2b-it): ')


In [ ]:
# 2. Load base google/gemma-2-2b-it (BF16) + Chewie 1.2 LoRA adapter, merge & save HF model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = 'google/gemma-2-2b-it'
MODEL_REV = '299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8'
ADAPTER_DIR = '/content/drive/MyDrive/MotherlyPulse_Chewie_1_2_Fresh/chewie_adapter'
MERGED_DIR = '/content/chewie_merged_hf'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REV,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get('HF_TOKEN'),
)
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
from huggingface_hub import hf_hub_download
import shutil
sp_path = hf_hub_download(repo_id=MODEL_ID, filename='tokenizer.model', revision=MODEL_REV, token=os.environ.get('HF_TOKEN'))
shutil.copy2(sp_path, os.path.join(MERGED_DIR, 'tokenizer.model'))
print(f'Merged HF model + tokenizer.model saved to {MERGED_DIR}')


In [ ]:
# 3. Clone & build llama.cpp (llama-quantize, llama-cli), convert to F16 GGUF, and quantize to Q4_K_M
!rm -rf /content/llama.cpp && git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize llama-cli -j4
!python /content/llama.cpp/convert_hf_to_gguf.py /content/chewie_merged_hf --outtype f16 --outfile /content/chewie-1.2-f16.gguf
!/content/llama.cpp/build/bin/llama-quantize /content/chewie-1.2-f16.gguf /content/chewie-1.2-q4_k_m.gguf Q4_K_M
!ls -lh /content/chewie-1.2-q4_k_m.gguf


In [ ]:
# 4. Sanity check chewie-1.2-q4_k_m.gguf with llama-cli
!/content/llama.cpp/build/bin/llama-cli \
  -m /content/chewie-1.2-q4_k_m.gguf \
  -cnv \
  -p "You are MotherlyPulse, a maternal and child health assistant for Ondo State, Nigeria. Start your reply with EMERGENCY:, PRIORITY:, or ROUTINE:. Reply in English.\n\nUser Question: I have a mild headache during pregnancy, what should I do?" \
  -n 100 --temp 0 --no-display-prompt


In [ ]:
# 5. Save chewie-1.2-q4_k_m.gguf and its SHA-256 checksum to Google Drive
import hashlib, os, shutil

DRIVE_DEST = '/content/drive/MyDrive/MotherlyPulse_Chewie_1_2_Fresh'
GGUF_SRC = '/content/chewie-1.2-q4_k_m.gguf'
GGUF_DST = os.path.join(DRIVE_DEST, 'chewie-1.2-q4_k_m.gguf')

shutil.copy2(GGUF_SRC, GGUF_DST)
sha256 = hashlib.sha256(open(GGUF_DST, 'rb').read()).hexdigest()
sha_file = GGUF_DST + '.sha256'
open(sha_file, 'w', encoding='utf-8').write(f'{sha256}  chewie-1.2-q4_k_m.gguf\n')
print(f'Saved: {GGUF_DST} ({os.path.getsize(GGUF_DST)} bytes)')
print(f'SHA-256: {sha256}')
